# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [1]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [2]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [3]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [4]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [6]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

In [7]:
import torch.nn.functional as F

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [8]:
# 1. item_emb — L2-нормалізовані жанрові вектори всіх книг
item_emb = F.normalize(item_feats, p=2, dim=1)

# Зберемо для кожного користувача його лайки у train: (item_idx, rating)
user_likes = defaultdict(list)
for u, b, rt in zip(train_pos["user_id"], train_pos["book_id"], train_pos["rating"]):
    user_likes[user_to_idx[u]].append((item_to_idx[b], float(rt)))

In [9]:
# 2. Вектор користувача — зважене (за оцінкою) середнє ембедингів уподобаних книг
def user_vector(user_idx):
    likes = user_likes.get(user_idx, [])
    if not likes:
        return torch.zeros(n_genres)
    idxs = torch.tensor([i for i, _ in likes])
    weights = torch.tensor([w for _, w in likes], dtype=torch.float32).unsqueeze(1)
    vec = (item_emb[idxs] * weights).sum(dim=0) / weights.sum()
    return F.normalize(vec, p=2, dim=0)

In [10]:
# 3. Оцінки (cosine) усіх книг для набору користувачів
def vsm_scores(user_idxs):
    U = torch.stack([user_vector(int(u)) for u in user_idxs])  
    return U @ item_emb.T                                       

In [11]:
vsm_recall = recall_at_k(vsm_scores, k=10)
print(f"Vector Space Model Recall@10: {vsm_recall:.4f}")

Vector Space Model Recall@10: 0.0515


In [12]:
# 4. Топ-5 рекомендацій для одного користувача (з назвами книг)
def recommend_vsm(user_idx, n=5):
    scores = vsm_scores(torch.tensor([user_idx]))[0].clone()
    for i in seen_by_user[user_idx]:
        scores[i] = -1e9                       # прибираємо вже прочитане
    top = torch.topk(scores, n).indices.tolist()
    return pd.DataFrame({
        "book": [title_of[items[i]] for i in top],
        "cosine": [round(float(scores[i]), 3) for i in top],
    })

In [13]:
print("Топ-5 рекомендацій для користувача 0:")
recommend_vsm(0, 5)

Топ-5 рекомендацій для користувача 0:


,book,cosine
0,"The Clan of the Cave Bear (Earth's Children, #1)",0.885
1,Stardust,0.885
2,The Alchemist,0.872
3,"Inkheart (Inkworld, #1)",0.868
4,Sophie's World,0.865


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


Recall@10 у векторного підходу низький, бо модель використовує тільки дуже прості жанрові ознаки книг. Вона не враховує індивідуальні патерни поведінки користувачів, популярність книг, авторів, схожість між користувачами та складніші приховані вподобання. Також жанри побудовані з користувацьких тегів, тому вони можуть бути шумними й занадто грубими. Через це модель може рекомендувати книги приблизно в тому ж жанрі, але не обов’язково саме ті, які користувач реально лайкнув би.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [14]:
class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32, temperature=10.0):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, emb_dim),
        )
        self.temperature = temperature

    def user_forward(self, user_idx):
        return F.normalize(self.user_emb(user_idx), p=2, dim=-1)

    def item_forward(self, feats):
        return F.normalize(self.item_tower(feats), p=2, dim=-1)

    def forward(self, user_idx, feats):
        u = self.user_forward(user_idx)
        v = self.item_forward(feats)
        return (u * v).sum(dim=-1) * self.temperature

In [15]:
def train_model(model, pos_u, pos_i, item_feats, n_epochs=30, lr=0.01, batch_size=4096, neg_ratio=4):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    n_pos = len(pos_u)

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_pos)
        batch_losses = []

        for start in range(0, n_pos, batch_size):
            idx = perm[start:start + batch_size]
            bu, bi = pos_u[idx], pos_i[idx]
            b = len(bu)

            # на кожен позитив — neg_ratio випадкових книг з усього корпусу
            neg_u = bu.repeat(neg_ratio)
            neg_i = torch.randint(0, M, (b * neg_ratio,))
            u_all = torch.cat([bu, neg_u])
            i_all = torch.cat([bi, neg_i])
            y_all = torch.cat([torch.ones(b), torch.zeros(b * neg_ratio)])

            # forward
            logits = model(u_all, item_feats[i_all])
            loss = criterion(logits, y_all)

            # backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        losses.append(float(np.mean(batch_losses)))
    return losses

In [16]:
two_tower = TwoTower(len(users), n_genres)
tt_losses = train_model(two_tower, pos_u, pos_i, item_feats, n_epochs=30, lr=0.01)
print(f"Two-Tower BCE loss: {tt_losses[0]:.4f} -> {tt_losses[-1]:.4f}")

Two-Tower BCE loss: 0.8338 -> 0.3888


In [17]:
def tt_scores(user_idxs):
    two_tower.eval()
    with torch.no_grad():
        item_vecs = two_tower.item_forward(item_feats)
        u = two_tower.user_forward(user_idxs)
        return (u @ item_vecs.T) * two_tower.temperature

In [18]:
print(f"Two-Tower Recall@10: {recall_at_k(tt_scores, k=10):.4f}")

Two-Tower Recall@10: 0.0834


In [19]:
def recommend_tt(user_idx, n=5):
    scores = tt_scores(torch.tensor([user_idx]))[0].clone()
    for i in seen_by_user[user_idx]:
        scores[i] = -1e9
    top = torch.topk(scores, n).indices.tolist()
    return pd.DataFrame({"book": [title_of[items[i]] for i in top]})

In [20]:
print("Топ-5 рекомендацій Two-Tower для користувача 0:")
recommend_tt(0, 5)

Топ-5 рекомендацій Two-Tower для користувача 0:


,book
0,"Angels & Demons (Robert Langdon, #1)"
1,Cloud Atlas
2,The Five People You Meet in Heaven
3,The Hunger Games Trilogy Boxset (The Hunger Ga...
4,"The Host (The Host, #1)"


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [21]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, user_idx, feats):
        u = self.user_emb(user_idx)
        x = torch.cat([u, feats], dim=-1)         
        return self.mlp(x).squeeze(-1)

In [22]:
# Навчаємо тим самим циклом і на тих самих позитивах/негативах, що й Two-Tower
ncf = NCF(len(users), n_genres)
ncf_losses = train_model(ncf, pos_u, pos_i, item_feats, n_epochs=30, lr=0.01)

print(f"NCF BCE loss: {ncf_losses[0]:.4f} -> {ncf_losses[-1]:.4f}")

NCF BCE loss: 0.5169 -> 0.3816


In [23]:
# Ранжування заданого списку кандидатів за sigmoid логіта
def rank_ncf(user_idx, candidate_idxs):
    ncf.eval()
    with torch.no_grad():
        u = torch.full((len(candidate_idxs),), user_idx, dtype=torch.long)
        feats = item_feats[torch.tensor(candidate_idxs)]
        scores = torch.sigmoid(ncf(u, feats))
    order = torch.argsort(scores, descending=True).tolist()
    return [(candidate_idxs[j], float(scores[j])) for j in order]

In [24]:
# Для порівняння з іншими моделями порахуємо Recall@10 (ранжуємо всі книги)
def ncf_scores(user_idxs):
    ncf.eval()
    out = []
    with torch.no_grad():
        for u in user_idxs:
            uu = torch.full((M,), int(u), dtype=torch.long)
            out.append(torch.sigmoid(ncf(uu, item_feats)))
    return torch.stack(out)

print(f"NCF Recall@10 (ранжування всіх книг): {recall_at_k(ncf_scores, k=10):.4f}")

NCF Recall@10 (ранжування всіх книг): 0.0649


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [25]:
# Етап 1: Retrieval — Two-Tower швидко відбирає топ-N кандидатів (без уже прочитаного)
def retrieve(user_idx, n_candidates=100):
    scores = tt_scores(torch.tensor([user_idx]))[0].clone()
    for i in seen_by_user[user_idx]:
        scores[i] = -1e9
    return torch.topk(scores, n_candidates).indices.tolist()

In [26]:
# Етап 2: Ranking — NCF точно переранжовує коротку добірку кандидатів.
def recommend_pipeline(user_idx, n_candidates=100, top_k=10):
    candidates = retrieve(user_idx, n_candidates)
    ranked = rank_ncf(user_idx, candidates)
    return ranked[:top_k]

In [27]:
# Демонстрація: що відібрав retrieval і що лишив ranking, для кількох користувачів
for user_idx in [0, 1, 5]:
    candidates = retrieve(user_idx, n_candidates=100)
    final = recommend_pipeline(user_idx, n_candidates=100, top_k=5)
    print(f"=== Користувач {user_idx} ===")
    print("Retrieval (топ-5 з 100 кандидатів Two-Tower):")
    for i in candidates[:5]:
        print("   ", title_of[items[i]])
    print("Ranking (фінальні топ-5 після NCF):")
    for i, s in final:
        print(f"    {title_of[items[i]]}  (p={s:.3f})")

=== Користувач 0 ===
Retrieval (топ-5 з 100 кандидатів Two-Tower):
    Angels & Demons  (Robert Langdon, #1)
    Cloud Atlas
    The Five People You Meet in Heaven
    The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
    The Host (The Host, #1)
Ranking (фінальні топ-5 після NCF):
    To Kill a Mockingbird  (p=0.567)
    Peace Like a River  (p=0.567)
    Angels & Demons  (Robert Langdon, #1)  (p=0.566)
    Kafka on the Shore  (p=0.555)
    The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)  (p=0.540)
=== Користувач 1 ===
Retrieval (топ-5 з 100 кандидатів Two-Tower):
    All the Light We Cannot See
    Orphan Train
    Sarah's Key
    Big Little Lies
    The Memory Keeper's Daughter
Ranking (фінальні топ-5 після NCF):
    The Goldfinch  (p=0.598)
    The White Tiger  (p=0.598)
    The Deep End of the Ocean (Cappadora Family, #1)  (p=0.598)
    The Story of Edgar Sawtelle  (p=0.598)
    Midwives  (p=0.598)
=== Користувач 5 ===
Retrieval (топ-5 з 100 кандидатів Two-Tower):
  

In [28]:
# --- Підсумкове порівняння всіх моделей ---
def pipeline_scores(user_idxs, n_candidates=200):
    out = torch.full((len(user_idxs), M), -1e9)
    for row, u in enumerate(user_idxs):
        for i, s in rank_ncf(int(u), retrieve(int(u), n_candidates)):
            out[row, i] = s
    return out

# Popularity baseline: рекомендуємо найпопулярніші книги (за кількістю оцінок у train).
pop_counts = train_df["book_id"].map(item_to_idx).value_counts()
pop = torch.tensor(pop_counts.reindex(range(M)).fillna(0).values, dtype=torch.float32)
def pop_scores(user_idxs):
    return pop.unsqueeze(0).repeat(len(user_idxs), 1)

In [29]:
summary = pd.DataFrame({
    "model": ["Popularity (baseline)", "VSM (content)", "Two-Tower", "NCF", "Pipeline (TT→NCF)"],
    "Recall@10": [
        recall_at_k(pop_scores, k=10),
        vsm_recall,
        recall_at_k(tt_scores, k=10),
        recall_at_k(ncf_scores, k=10),
        recall_at_k(lambda uu: pipeline_scores(uu, 200), k=10),
    ],
}).sort_values("Recall@10", ascending=False).reset_index(drop=True)
summary

,model,Recall@10
0,Popularity (baseline),0.097378
1,Two-Tower,0.083413
2,Pipeline (TT→NCF),0.069638
3,NCF,0.064877
4,VSM (content),0.051546


**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Ділимо на два етапи, бо NCF повільніше рахує оцінку для кожної пари user–book. Якщо книг дуже багато, проганяти NCF одразу по всіх книгах дорого. Тому спочатку Two-Tower швидко відбирає невелику кількість кандидатів, а потім NCF точніше їх переранжовує. Це робить систему швидшою і практичнішою для реальних великих рекомендаційних систем.

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


**Відповіді:**

1. Усі моделі показали доволі низький Recall@10 (приблизно від 0.05 до 0.10), причинами чого може бути:
- Ми використовували дуже бідні контентні ознаки книг — лише 12 жанрів, побудованих із тегів. Дві книги одного жанру можуть сильно відрізнятися за сюжетом, стилем, автором чи цільовою аудиторією, але модель цього не бачить.
- Матриця взаємодій є дуже розрідженою. Навіть після підвибірки маємо близько 141 тисячі взаємодій на майже 3 мільйони можливих пар користувач-книга. Більшість книг кожен користувач ніколи не оцінював.
- Валідаційні лайки не містять усіх книг, які потенційно могли б сподобатися користувачу. Якщо модель рекомендує хорошу книгу, яку користувач просто не встиг оцінити, така рекомендація все одно вважається помилкою.
- Також цікавий момент: простий popularity-baseline б'є всі навчені моделі. Популярні книги частіше потрапляють у val-лайки, тож вгадати популярне — сильна стратегія саме для recall, хоча персоналізації в ній нуль.
       
2. Не змінюючи саму модель, можна значно покращити ознаки:
- Метадані книг із books.csv: автор (ембединг автора — дуже сильний сигнал смаку), рік видання, мова, average_rating та кількість оцінок (проксі якості/популярності), серія.
- Повний набір тегів, а не 12 жанрів: взяти сотні топ-тегів і закодувати їх через TF-IDF по book_tags (з вагою count) — це різко підвищить роздільну здатність item-простору.
- Текстові ембединги назви/опису через BERT — семантика, якої жанри не передають.
- Ознаки користувача: його середній рейтинг/строгість, активність, улюблені автори/жанри як агрегати історії.
- Поведінкові/часові ознаки: свіжість і послідовність взаємодій.
            
3. Якщо користувач любить фентезі, десять фентезі-книг підряд можуть виглядати релевантно за скором, але створюють одноманітну стрічку й швидко знижують корисність рекомендацій.
Технічно різноманітність можна додати через reranking: штрафувати надто схожі між собою айтеми, обмежувати кількість книг одного жанру/автора/серії в top-K, використовувати MMR або змішувати частину слотів з суміжних жанрів.
             
4. Нова книга без оцінок створює проблему cold start, оскільки модель не має історії взаємодій користувачів із цією книгою.
- Vector Space Model (VSM) може рекомендувати таку книгу одразу, якщо для неї відомі жанри або інші контентні ознаки. Модель порівнює контент книги з уподобаннями користувача і не потребує попередніх оцінок.
- Two-Tower також може працювати з новою книгою. Достатньо побудувати її ознаки та пропустити через Item Tower, щоб отримати ембединг книги. Після цього її можна знаходити через similarity-пошук разом з іншими книгами.
- NCF у нашій реалізації обробляє книгу тільки через жанрові ознаки (без item-ембедингу), тому нову книгу з відомими жанрами він оцінює нарівні з будь-якою іншою книгою такого ж жанрового профілю — окремої проблеми cold start для айтема тут немає. Натомість у класичному NCF, де кожна книга має власний навчений ембединг, нова книга справді страждала б від cold start, бо її ембединг не натренований.     
- Найгірше проблему cold start переносять чисто колаборативні підходи, які покладаються лише на історію взаємодій користувачів і книг. Саме тому контентні ознаки (жанри, теги, опис, автор тощо) є важливими для рекомендації нових об’єктів.
             
5. CTR оптимізує клік, але клік може бути наслідком клікбейту: користувач відкрив відео й одразу закрив. Watch time краще наближує справжню корисність, бо показує, що користувач залишився дивитися. У weighted logistic regression це можна вшити через вагу прикладу: позитивні взаємодії з довшим переглядом отримують більшу вагу в loss, а короткі або випадкові кліки меншу. Тоді модель не просто вчиться передбачати факт кліку, а сильніше підлаштовується під взаємодії, які дали більше часу перегляду.